<div style="border-left:6px solid #00A982;padding:10px 18px;margin-bottom:18px">
<h1 style="margin:0">Session 1 &mdash; Programming with QoolQit &mdash; Hands-on</h1>
<p style="margin:6px 0 0;color:#52666d"><strong>Guided exercise</strong> &middot; From atom positions to Rabi oscillations and the Rydberg blockade</p>
</div>

In this notebook you build, compile and run your first neutral-atom program, and you use it to
observe the **Rydberg blockade**: two atoms driven by the *same* pulse behave completely
differently depending only on how far apart you place them.

### Learning objectives

By the end of this notebook you will be able to:

- import the core QoolQit components needed to build a program;
- define a neutral-atom **Register** from coordinates;
- configure an analog **Drive** (pulse sequence);
- pair register and drive into a **QuantumProgram** and compile it to a device;
- run the compiled program on a **local emulator**;
- inspect and interpret measurement results, using the **Rydberg blockade** as a concrete example.

### How to work through this notebook

The hands-on cells are partially written. Wherever you see `...` you have to fill something in,
and the comment right above tells you what. The tasks are labelled:

| Label | Meaning |
|---|---|
| **Task** | fill in the `...` to make the cell run |
| **Question** | no code needed &mdash; write your answer in the markdown cell below |
| **Bonus** | optional, do it if you have time left |

Cells marked *given* are complete &mdash; just run them. Work top to bottom: later cells reuse the
variables you define earlier, so a cell you skipped will make the next one fail with a
`NameError` or an `Ellipsis`-related `TypeError`.

<b>Hints</b> are hidden in collapsible blocks &mdash; open them only after you have tried.

----

# The physics we want to see

We build two versions of the *same* two-qubit program, keeping the drive identical and changing
only the **distance** between the atoms.

| Case | Distance | Interaction $\tilde{J}$ | Regime | Expected behaviour |
|------|----------|--------------------------|--------|--------------------|
| Far apart | $3.0$ | $\approx 0.0014 \ll \tilde{\Omega}$ | Free Rabi oscillation | Both atoms independently flip; $\lvert 11\rangle$ fully accessible |
| Close together | $1.0$ (minimum) | $= 1 \gg \tilde{\Omega}$ | Rydberg blockade | Simultaneous double excitation suppressed |

The steps are: import components, define the registers, configure the drive, build and compile
the programs, run them on a local emulator, then inspect the results.

The cell below imports everything you need and fixes the plotting colours. This is *given*.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from qoolqit.devices import AnalogDevice
from qoolqit.execution import BitStrings, EmulationConfig, LocalEmulator
from qoolqit.waveforms import ConstantWaveform

PASQAL_GREEN = "#00A982"
PASQAL_DARK = "#102A33"
PASQAL_GRAY = "#C9D8D4"


---
## Step 1 &mdash; Define the registers

A `Register` fixes the qubit positions. By convention, unit distance corresponds to interaction
strength $\tilde{J} = 1$; interactions scale as $\tilde{J} \propto 1/r^6$, so moving the atoms
apart or together changes the interaction by orders of magnitude.

**Task 1.1** &mdash; build two two-atom registers from explicit coordinates:

- `register_far`: atoms at $(0,0)$ and $(3.0, 0)$;
- `register_close`: atoms at $(0,0)$ and $(1.0, 0)$ &mdash; the minimum distance the device allows.

Then print the interaction of each register and draw both.

<details><summary>Hint</summary>

`Register.from_coordinates` takes a list of `(x, y)` tuples, e.g.
`Register.from_coordinates([(0, 0), (3.0, 0)])`.

`register.interactions()` returns a dict mapping a qubit pair to its interaction strength.
Each register has a `.draw(ax=...)` method, so you can send the two plots to the two
pre-created axes `ax1` and `ax2`.
</details>

In [ ]:
from qoolqit import Register

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Task 1.1a: two atoms 3.0 units apart, and two atoms 1.0 unit apart.
register_far = ...
register_close = ...

# Task 1.1b: print the interaction strength of each register.
print("Far apart      -> J =", ...)
print("Close together -> J =", ...)

# Task 1.1c: draw each register on its own axis.
...
...


**Question 1.2** &mdash; the distance only changed by a factor of $3$, yet the interaction changed by
a factor of roughly $700$. Where does that number come from?

---
## Step 2 &mdash; Configure the analog pulse sequence

We drive both qubits with the same **constant-amplitude** pulse and zero detuning &mdash; a plain
Rabi drive. Whether the two atoms actually behave independently depends entirely on the register
we pair it with.

**Task 2.1** &mdash; build a `Drive` with

- `duration = 100` (dimensionless time units);
- a **constant** amplitude $\tilde{\Omega} = 0.2$;
- a **constant** detuning equal to $0$.

Then draw it.

<details><summary>Hint</summary>

`ConstantWaveform(duration, value)` is already imported. A `Drive` takes the keyword arguments
`amplitude=` and `detuning=`, each a waveform. Note that $\tilde{\Omega} = 0.2$ is much smaller
than the close-register interaction $\tilde{J} = 1$ &mdash; that separation of scales is exactly what
produces the blockade.
</details>

In [ ]:
from qoolqit import Drive

duration = 100
omega = 0.2

# Task 2.1: constant amplitude omega, zero detuning, both lasting `duration`.
drive = Drive(
    amplitude=...,
    detuning=...,
)

drive.draw()


---
## Step 3 &mdash; Build and compile the programs

A `QuantumProgram` pairs a register with a drive. `compile_to` maps it onto a specific device
model &mdash; here the default `AnalogDevice` &mdash; translating the dimensionless quantities into
physical units (ns, rad/µs, µm) and checking every hardware constraint on the way.

**Task 3.1** &mdash; pair each register with the *same* drive, then compile both programs to an
`AnalogDevice()` instance.

<details><summary>Hint</summary>

`QuantumProgram(register, drive)` takes the register first. Create the device once
(`device = AnalogDevice()`) and call `program.compile_to(device)` on each program. Reusing one
device object means both programs get the same physical scaling, so the comparison in step 5 is
fair.
</details>

In [ ]:
from qoolqit import QuantumProgram

# Task 3.1a: one program per register, both using the drive from step 2.
program_far = ...
program_close = ...

# Task 3.1b: instantiate the device and compile both programs onto it.
device = ...
...
...


Run the cell below (*given*) to look at the device specifications.

In [ ]:
device.specs


**Question 3.2** &mdash; compare `max_amplitude` in the specs with the $\tilde{\Omega} = 0.2$ you asked
for, and `min_distance` with the distances you used. Which of your two registers sits exactly at
a hardware limit, and what would happen if you tried an amplitude of $0.5$?

---
## Step 4 &mdash; Run on the local emulator

We sample bitstrings at several evaluation times along the drive so we can track the Rabi
oscillation, not just the final state.

**Task 4.1** &mdash; set up the emulation and run both programs:

- 41 evaluation times evenly spaced on the *relative* interval $[0, 1]$;
- `num_shots = 500`;
- wrap them in a `BitStrings` observable, put that observable into an `EmulationConfig`, and
  hand the configuration to a `LocalEmulator`;
- run each program and collect its results.

<details><summary>Hint</summary>

`np.linspace(0.0, 1.0, 41)` gives the evaluation times; `BitStrings` expects them as a plain
list, so use `list(eval_times)`.

`EmulationConfig(observables=[...])` takes a *list* of observables, and
`LocalEmulator(emulation_config=configuration)` takes the configuration.

`emulator.run(program)` returns a job; `job.results()` gives the sampled data. The `assert` lines
check that the job reached `JobStatus.DONE` before you read it &mdash; leave them in.
</details>

In [ ]:
# Task 4.1a: 41 evaluation times between 0 and 1, and 500 shots per time.
eval_times = ...
num_shots = ...

# Task 4.1b: the observable, the configuration and the emulator.
bitstrings = BitStrings(evaluation_times=..., num_shots=...)
configuration = EmulationConfig(observables=...)

emulator = ...

# Task 4.1c: run the far-apart program and read its results.
job_far = ...
results_far = ...

# Task 4.1d: same for the close-together program.
job_close = ...
results_close = ...


---
## Step 5 &mdash; Inspect and interpret the results

We track $P_{11}(t)$, the probability of finding **both** atoms excited, as a function of time.
Free Rabi oscillation should let $P_{11}$ reach $1$; the blockade should keep it close to $0$.

**Task 5.1** &mdash; for each of the two runs, extract the evaluation times and the probability of the
bitstring `"11"` at each of them.

<details><summary>Hint</summary>

`results.get_result_times(bitstrings)` returns the list of times at which the observable was
evaluated.

`results.get_result(bitstrings.tag, time=t)` returns a counter of bitstrings for that time. Use
`.get("11", 0)` so that a time where `"11"` was never sampled counts as zero instead of raising,
and divide by `num_shots` to turn counts into a probability.

A list comprehension over `t_far` is the compact way to build `p11_far`.
</details>

In [ ]:
# Task 5.1a: the times at which the far-apart run was sampled, and P_11 at each of them.
t_far = ...
p11_far = [
    ...
    for t in t_far
]

# Task 5.1b: the same two quantities for the close-together run.
t_close = ...
p11_close = [
    ...
    for t in t_close
]


The plotting cell below is *given* &mdash; run it once `t_far`, `p11_far`, `t_close` and `p11_close`
are filled in.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(t_far, p11_far, "-", color=PASQAL_GREEN, lw=2, label="Far apart (free Rabi)")
ax.plot(
    t_close, p11_close, "-", color=PASQAL_DARK, lw=2, label="Close together (blockade)"
)
ax.set_xlabel("dimensionless time $t$")
ax.set_ylabel(r"$P_{11}$ (both atoms excited)")
ax.set_ylim(-0.05, 1.05)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(alpha=0.18)
ax.set_title("Rabi oscillations", loc="left", weight="bold")
ax.legend()
fig.tight_layout()
plt.show()

print(f"Far apart:      max P_11 = {max(p11_far):.3f}")
print(f"Close together: max P_11 = {max(p11_close):.3f}")


**Question 5.2** &mdash; describe the two curves. For the far-apart register, what is the period of
the $P_{11}$ oscillation compared to the single-atom Rabi period $2\pi/\tilde{\Omega}$, and why?
For the close register, why does $P_{11}$ stay near zero even though the pulse is long enough to
excite each atom many times over?

---
### Recap

You have written a complete QoolQit program: a **register** (geometry, hence interactions), a
**drive** (the pulse sequence), compiled into a device-specific **program** and executed on an
**emulator**. Changing a single number &mdash; the distance &mdash; switched the system between two
qualitatively different physical regimes, which is exactly the lever we will use in session 2 to
encode optimization problems into atom positions.